<a href="https://colab.research.google.com/github/krishuynh2222/retail-end-to-end-analytics/blob/main/01_notebooks/02_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 02: Data Cleaning
##### **PURPOSE:**
###### Fix all data quality issues found in 01_data_audit.jpynb
###### Each table has its own dedicated cell so we can rerun one table
###### without touching the others

##### **HOW TO USE THIS NOTEBOOK:**
###### Run cells top to bottom the first time
###### If one table fails the quality check in Notebook 03:
###### 1. Fix the problem in that table's cell below
###### 2. Re-run that single cell only
###### 3. Go back to Notebook 03 and re-run it
###### WE do NOT need to re-run all 11 cells every time.

##### **INPUT**
###### datasets/    11 raw CSV fles (must keep raw datasets)

##### **OUTPUT FIELDS** (saved to GGDrive)
######   datasets_clean/      11 clean CSV files (ready for BigQuery)
######   audit/02_cleaning_log.txt       every change made, with timestamps
######   audit/02_cleaning_summary.json  row counts and null percentages after cleaning


## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Configure Paths

In [ ]:
import os

RAW_DIR   = '/content/drive/MyDrive/AmazinChoices/datasets'
CLEAN_DIR = '/content/drive/MyDrive/AmazinChoices/datasets_clean'
AUDIT_DIR = '/content/drive/MyDrive/AmazinChoices/audit'

os.makedirs(CLEAN_DIR, exist_ok=True)
os.makedirs(AUDIT_DIR, exist_ok=True)

print("RAW_DIR:   " + RAW_DIR)
print("CLEAN_DIR: " + CLEAN_DIR)
print("AUDIT_DIR: " + AUDIT_DIR)

RAW_DIR:   /content/drive/MyDrive/AmazinChoices/datasets
CLEAN_DIR: /content/drive/MyDrive/AmazinChoices/datasets_clean
AUDIT_DIR: /content/drive/MyDrive/AmazinChoices/audit


## Import libraries

In [ ]:
import pandas as pd
import numpy as np
import json
import logging
import warnings
from datetime import datetime

warnings.filterwarnings('ignore')

In [ ]:
# Remove existing handlers so the log file resets cleanly when we re-run
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

LOG_PATH = os.path.join(AUDIT_DIR, '02_cleaning_log.txt')

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(message)s",
    datefmt="%H:%M:%S",
    handlers=[
        logging.FileHandler(LOG_PATH, mode='w', encoding='utf-8'),
        logging.StreamHandler()
    ]
)

log = logging.getLogger()

print("Logger ready. Log file: " + LOG_PATH)


Logger ready. Log file: /content/drive/MyDrive/AmazinChoices/audit/02_cleaning_log.txt


## Define functions
###### parse_date()       handles mixed date formats in a single column
###### clean_currency()   handles mixed currency formats (dollar sign, EU decimal, etc.)
###### normalize_email()  lowercases emails so JOIN keys work correctly

In [ ]:
def parse_date(series, col_name):
    """
    Parse a date column that contains multiple different formats at the same time.

    Why pd.to_datetime() alone is not enough:
      In the raw data, the same date column can contain all of these at once:
        '2025-01-08T03:21:49-08:00'   ISO format with timezone offset
        '2025-02-07T14:41:15Z'        ISO format in UTC
        '10/23/25'                    US short format: month/day/year
        '3-Nov-25'                    Day, month abbreviation, year
        '14-04-2025'                  European format: day-month-year

      pd.to_datetime() can parse ISO format but fails on the others.
      This function first tries pd.to_datetime() with utc=True,
      then loops through a list of explicit formats to catch what remains.

    Arguments:
      series    a pandas Series containing the raw date strings
      col_name  a string used only for logging, so you know which column

    Returns:
      A pandas Series of datetime objects in UTC timezone.
      Values that could not be parsed are returned as NaT (Not a Time).
    """
    original_null_count = series.isna().sum()

    parsed = pd.to_datetime(series, utc=True, errors='coerce')

    explicit_formats = [
        "%m/%d/%y",
        "%m/%d/%Y",
        "%d-%b-%y",
        "%d-%b-%Y",
        "%d-%m-%Y",
        "%Y-%m-%d",
        "%m-%d-%Y",
    ]

    for date_format in explicit_formats:
        still_null = parsed.isna() & series.notna()
        if still_null.sum() == 0:
            break
        parsed.loc[still_null] = pd.to_datetime(
            series[still_null],
            format=date_format,
            errors='coerce',
            utc=True
        )

    new_nulls = parsed.isna().sum() - original_null_count
    if new_nulls > 0:
        message = col_name + ": " + str(new_nulls) + " values could not be parsed and became NaT"
        log.warning(message)
        print("  WARNING " + message)
    else:
        print("  OK      " + col_name + ": all dates parsed successfully")

    return parsed

In [ ]:
def clean_currency(series, col_name):
    """
    Convert messy currency strings to plain float numbers.

    Why a simple float() conversion does not work:
      The same price column can contain all of these at once:
        '$8.84'      has a dollar sign that must be stripped
        'USD 15.00'  has a currency code and a space
        '7,73'       EU decimal format where comma means decimal point (= 7.73)
        '1,347'      US thousands separator where comma separates groups (= 1347)
        '$1.78 '     trailing space

      The difficult case is the comma.
        '7,73'   has exactly 2 digits after the comma, so it is EU decimal
        '1,347'  has exactly 3 digits after the comma, so it is US thousands
      We handle these differently based on that digit count rule.

    Arguments:
      series    a pandas Series containing the raw currency strings
      col_name  a string used only for logging

    Returns:
      A pandas Series of float values.
      Values that could not be converted are returned as NaN.
    """
    # Step 1: strip everything that is not a digit, period, comma, or minus sign
    cleaned = series.astype(str).str.strip()
    cleaned = cleaned.str.replace(r'[^\d.,\-]', '', regex=True)

    # Step 2: detect EU decimal format (exactly 2 digits after comma, e.g. '7,73')
    eu_decimal_mask = cleaned.str.match(r'^\d+,\d{2}$')
    cleaned.loc[eu_decimal_mask] = cleaned.loc[eu_decimal_mask].str.replace(',', '.', regex=False)

    # Step 3: remove remaining commas (US thousands separators like '1,347')
    cleaned = cleaned.str.replace(',', '', regex=False)

    # Step 4: convert to float
    result = pd.to_numeric(cleaned, errors='coerce')

    new_nulls = result.isna().sum() - series.isna().sum()
    if new_nulls > 0:
        message = col_name + ": " + str(new_nulls) + " values could not be converted and became NaN"
        log.warning(message)
        print("  WARNING " + message)

    return result

In [ ]:
def normalize_email(series):
    """
    Convert email addresses to lowercase and remove leading/trailing whitespace.

    Why this is the most important fix in the entire pipeline:
      Email is the JOIN key between o rders and customer_profiles.
      'ELLA@GMAIL.COM' and 'ella@gmail.com' are the same person.
      Without this fix, a JOIN between the two tables will silently fail
      to match that row. You lose that customer's data with no error message.
      Every customer metric (retention, LTV, churn) will be wrong.
    """
    return series.str.lower().str.strip()

## Clean orders
###### This is the central table. Every other table connects to it via order_id
###### or customer_email, so it must be cleaned first.
###### ========================================================================
######   **PROBLEM 1:** order_timestamp has 5 mixed date formats
######   - FIX:       parse_date() handles all formats and converts to UTC datetime
######   - NEW COLS:  order_date (date only), order_year_month (YYYY-MM string)
######   **PROBLEM 2:** customer_email has ALL CAPS values like 'ELLA@GMAIL.COM'
######   - FIX:       normalize_email() converts everything to lowercase
######   - WHY:       this is the JOIN key to customer_profiles, must match exactly
######   **PROBLEM 3:** shipping_fee_raw has currency symbols and negative values
######   - FIX:       clean_currency() removes symbols, .clip(lower=0) removes negatives
######   - WHY:      shipping fee cannot be negative, it was a data entry error
######   **PROBLEM 4:** tax_raw has currency symbols and negative values
######   - FIX:       same as shipping_fee
######   **PROBLEM 5:** currency_raw has variants like 'Usd', 'US D', '$', 'usd'
######   - FIX:       overwrite entire column with the string 'USD'
######   - WHY:       all orders are in USD, these are just inconsistent spellings
######   **PROBLEM 6:** promo_code_raw has leading/trailing whitespace and 'NONE' strings
######   - FIX:       .str.strip().str.upper() and replace 'NONE' and '' with NaN
#######   **WHY:**       'NONE' as a string would be counted as a real promo code in GROUP BY
######   **PROBLEM 7:** platform and order_status have inconsistent casing
######   - FIX:       .str.strip().str.title() for consistent grouping

In [ ]:
df_raw = pd.read_csv(os.path.join(RAW_DIR, 'orders.csv'))
log.info("orders  raw: " + str(len(df_raw)) + " rows, " + str(df_raw.shape[1]) + " columns")
print("Raw: " + str(len(df_raw)) + " rows")

22:43:35  orders  raw: 110000 rows, 11 columns


Raw: 110000 rows


In [ ]:
out = df_raw.copy()

# Fix 1: dates
out['order_timestamp']  = parse_date(df_raw['order_timestamp'], 'order_timestamp')
out['order_date']       = out['order_timestamp'].dt.date
out['order_year_month'] = out['order_timestamp'].dt.to_period('M').astype(str)

# Fix 2: email
all_caps_count = df_raw['customer_email'].str.isupper().sum()
out['customer_email'] = normalize_email(df_raw['customer_email'])
log.info("orders  customer_email: " + str(all_caps_count) + " ALL CAPS values converted to lowercase")
print("  OK      customer_email: " + str(all_caps_count) + " ALL CAPS converted to lowercase")

# Fix 3: shipping_fee
out['shipping_fee'] = clean_currency(df_raw['shipping_fee_raw'], 'shipping_fee_raw')
neg_count = (out['shipping_fee'] < 0).sum()
if neg_count > 0:
    log.warning("orders  shipping_fee: " + str(neg_count) + " negative values clipped to 0")
    print("  WARNING shipping_fee: " + str(neg_count) + " negative values clipped to 0")
out['shipping_fee'] = out['shipping_fee'].clip(lower=0)

# Fix 4: tax
out['tax'] = clean_currency(df_raw['tax_raw'], 'tax_raw')
neg_tax_count = (out['tax'] < 0).sum()
if neg_tax_count > 0:
    log.warning("orders  tax: " + str(neg_tax_count) + " negative values clipped to 0")
    print("  WARNING tax: " + str(neg_tax_count) + " negative values clipped to 0")
out['tax'] = out['tax'].clip(lower=0)

# Fix 5: currency
currency_variants = df_raw['currency_raw'].value_counts().to_dict()
log.info("orders  currency variants found: " + str(currency_variants) + " -> all set to USD")
print("  OK      currency variants " + str(currency_variants) + " -> all set to USD")
out['currency'] = 'USD'

# Fix 6: promo_code
out['promo_code'] = (
    df_raw['promo_code_raw']
    .str.strip()
    .str.upper()
    .replace({'': np.nan, 'NONE': np.nan})
)
print("  OK      promo_code: whitespace stripped, NONE string converted to NaN")

# Fix 7: platform and order_status
out['platform']     = df_raw['platform'].str.strip().str.title()
out['order_status'] = df_raw['order_status'].str.strip().str.title()
print("  OK      platform values: " + str(sorted(out['platform'].unique())))
print("  OK      order_status values: " + str(sorted(out['order_status'].unique())))

# Drop the original raw columns that have been replaced
out.drop(columns=['currency_raw', 'order_timestamp', 'shipping_fee_raw', 'tax_raw', 'promo_code_raw'], inplace=True)

out.to_csv(os.path.join(CLEAN_DIR, 'orders.csv'), index=False)
orders_clean = out

log.info("orders  clean: " + str(len(out)) + " rows saved")
print("")
print("Saved: " + CLEAN_DIR + "/orders.csv")
print("Clean: " + str(len(out)) + " rows")
print("Null%  shipping_fee=" + str(round(out['shipping_fee'].isna().mean() * 100, 1)) + "%"
      + "  tax=" + str(round(out['tax'].isna().mean() * 100, 1)) + "%"
      + "  promo_code=" + str(round(out['promo_code'].isna().mean() * 100, 1)) + "% (expected, not all orders use promos)")


22:43:35  orders  customer_email: 8934 ALL CAPS values converted to lowercase


  OK      order_timestamp: all dates parsed successfully
  OK      customer_email: 8934 ALL CAPS converted to lowercase


22:43:36  orders  shipping_fee: 1565 negative values clipped to 0


  WARNING shipping_fee: 1565 negative values clipped to 0


22:43:36  orders  tax: 1622 negative values clipped to 0
22:43:36  orders  currency variants found: {'Usd': 22200, 'US D': 22155, 'USD': 21922, '$': 21905, 'usd': 21818} -> all set to USD


  WARNING tax: 1622 negative values clipped to 0
  OK      currency variants {'Usd': 22200, 'US D': 22155, 'USD': 21922, '$': 21905, 'usd': 21818} -> all set to USD
  OK      promo_code: whitespace stripped, NONE string converted to NaN
  OK      platform values: ['Amazon', 'Facebook', 'Instagram', 'Local', 'Tiktok', 'Walmart', 'Website']
  OK      order_status values: ['Cancelled', 'Completed', 'Pending', 'Refunded', 'Shipped']


22:43:40  orders  clean: 110000 rows saved



Saved: /content/drive/MyDrive/AmazinChoices/datasets_clean/orders.csv
Clean: 110000 rows
Null%  shipping_fee=11.9%  tax=11.8%  promo_code=32.6% (expected, not all orders use promos)


## Clean order_items
###### This is the central table. Every other table connects to it via order_id or customer_email, so it must be cleaned first.
###### ========================================================================
######   **PROBLEM 1:** item_price_raw and discount_raw have currency symbols and EU decimals
######   - FIX:       clean_currency() handles all formats, .clip(lower=0) removes negatives

######   **PROBLEM 2:** discount_amt is larger than item_price on some rows
######   - FIX:       cap discount_amt at item_price (a discount cannot exceed the price)
######   - WHY:  if discount > price then (price - discount) is negative, which makes line_revenue negative and corrupts total revenue

######   **PROBLEM 3:** qty_raw is stored as a string and some values are zero or negative
######   - FIX: convert to integer, set zero and negative values to NaN
######   - WHY:      quantity must be at least 1. A sale of 0 items or -1 items is invalid.

######   **PROBLEM 4:** sku_raw has inconsistent casing
######   - FIX: .str.upper() so it matches product_master and cost_history

######   **PROBLEM 5:** no line_revenue column exists in the raw data
######   - NEW COL:   line_revenue = (item_price - discount_amt) * qty
######   - WHY: this is the core revenue metric for every downstream analysis

In [ ]:
df_raw = pd.read_csv(os.path.join(RAW_DIR, 'order_items.csv'))
log.info("order_items  raw: " + str(len(df_raw)) + " rows")
print("Raw: " + str(len(df_raw)) + " rows")

23:25:49  order_items  raw: 130000 rows


Raw: 130000 rows


In [ ]:
out = df_raw.copy()

#item_price and discount_amt
out['item_price']   = clean_currency(df_raw['item_price_raw'], 'item_price_raw').clip(lower=0)
out['discount_amt'] = clean_currency(df_raw['discount_raw'],   'discount_raw').clip(lower=0)

#cap discount at item_price
excess_count = (out['discount_amt'].fillna(0) > out['item_price'].fillna(0)).sum()
if excess_count > 0:
    log.warning("order_items  discount_amt: " + str(excess_count) + " rows where discount > price, discount capped at price")
    print("  WARNING discount_amt: " + str(excess_count) + " rows where discount > price, capped at item_price")
    mask = out['discount_amt'].fillna(0) > out['item_price'].fillna(0)
    out.loc[mask, 'discount_amt'] = out.loc[mask, 'item_price']

#qty
out['qty'] = pd.to_numeric(df_raw['qty_raw'], errors='coerce')
# Build mask BEFORE fillna so zeros and negatives are caught correctly
bad_qty_mask  = (out['qty'] <= 0) | (out['qty'].isna())
bad_qty_count = (out['qty'] <= 0).sum()   # count non-null bad values only
if bad_qty_count > 0:
    log.warning("order_items  qty: " + str(bad_qty_count) + " values <= 0 set to NaN")
    print("  WARNING qty: " + str(bad_qty_count) + " zero or negative values set to NaN")
    out.loc[out['qty'] <= 0, 'qty'] = np.nan  # use np.nan, not pd.NA

# Convert to float (not Int64) — avoids nullable integer edge cases
out['qty'] = out['qty'].astype(float)

#sku and product_name
out['sku']          = df_raw['sku_raw'].str.strip().str.upper()
out['product_name'] = df_raw['product_name_raw'].str.strip().str.title()
print("  OK      sku: converted to uppercase")
print("  OK      product_name: converted to Title Case")

# New column: line_revenue
out['line_revenue'] = (out['item_price'] - out['discount_amt'].fillna(0)) * out['qty']
neg_rev_count = (out['line_revenue'] < 0).sum()
if neg_rev_count > 0:
    log.warning("order_items  line_revenue: " + str(neg_rev_count) + " negative values, flagged but kept for investigation")
    print("  WARNING line_revenue: " + str(neg_rev_count) + " negative values, kept for investigation")

out.drop(columns=['sku_raw', 'product_name_raw', 'qty_raw', 'item_price_raw', 'discount_raw'], inplace=True)

out.to_csv(os.path.join(CLEAN_DIR, 'order_items.csv'), index=False)
order_items_clean = out

log.info("order_items  clean: " + str(len(out)) + " rows, total revenue=" + str(round(out['line_revenue'].sum())))
print("")
print("Saved: " + CLEAN_DIR + "/order_items.csv")
print("Clean: " + str(len(out)) + " rows")
print("Total line_revenue: $" + str(round(out['line_revenue'].sum())))

23:34:37  order_items  discount_amt: 12195 rows where discount > price, discount capped at price


  WARNING discount_amt: 12195 rows where discount > price, capped at item_price
  OK      sku: converted to uppercase
  OK      product_name: converted to Title Case


23:34:38  order_items  clean: 130000 rows, total revenue=5413551



Saved: /content/drive/MyDrive/AmazinChoices/datasets_clean/order_items.csv
Clean: 130000 rows
Total line_revenue: $5413551


## Clean customer_profiles

######   **PROBLEM 1:** customer_email has ALL CAPS values
######   - FIX:  normalize_email() converts to lowercase
######   - WHY:  this is the JOIN key to orders, must match exactly

######   **PROBLEM 2:** first_purchase_date has mixed date formats
######   - FIX: parse_date() handles all formats

######   **PROBLEM 3:** state has inconsistent casing
######   - FIX: .str.upper() because state codes are always uppercase (CA, NY, TX)

######   **PROBLEM 4:** some rows have segment = 'Wholesale?'
######   --> DECISION:  keep these rows exactly as they are, do not fix or drop
######   - WHY: this is a business question, not a data entry error we can fix.
######   It needs a business person to confirm: are these real wholesale customers or a mistake? We flag it and document it.

######   **PROBLEM 5:** duplicate customer_email rows exist
######   - FIX: keep the first occurrence, drop the rest
######   - WHY: customer_profiles is a dimension table (one row per customer).
######   --> Duplicates would cause many-to-many JOINs, inflating row counts.

In [ ]:
df_raw = pd.read_csv(os.path.join(RAW_DIR, 'customer_profiles.csv'))
log.info("customer_profiles  raw: " + str(len(df_raw)) + " rows")
print("Raw: " + str(len(df_raw)) + " rows")

22:43:46  customer_profiles  raw: 80000 rows


Raw: 80000 rows


In [ ]:
out = df_raw.copy()

out['customer_email']      = normalize_email(df_raw['customer_email'])
out['first_purchase_date'] = parse_date(df_raw['first_purchase_date'], 'first_purchase_date')
out['segment']             = df_raw['segment'].str.strip()
out['state']               = df_raw['state'].str.strip().str.upper()

wholesale_count = (out['segment'] == 'Wholesale?').sum()
if wholesale_count > 0:
    log.warning("customer_profiles  segment: " + str(wholesale_count) + " rows have Wholesale? value, kept as-is, needs business confirmation")
    print("  WARNING segment: " + str(wholesale_count) + " rows have segment = 'Wholesale?'")
    print("          These are kept as-is. A business person must confirm whether")
    print("          these are real wholesale customers or a data entry mistake.")

rows_before_dedup = len(out)
out.drop_duplicates(subset='customer_email', keep='first', inplace=True)
rows_removed = rows_before_dedup - len(out)
if rows_removed > 0:
    log.warning("customer_profiles  customer_email: " + str(rows_removed) + " duplicate rows removed, first occurrence kept")
    print("  WARNING customer_email: " + str(rows_removed) + " duplicate rows removed (kept first occurrence)")

out.to_csv(os.path.join(CLEAN_DIR, 'customer_profiles.csv'), index=False)
customer_profiles_clean = out

log.info("customer_profiles  clean: " + str(len(out)) + " rows saved")
print("")
print("Saved: " + CLEAN_DIR + "/customer_profiles.csv")
print("Clean: " + str(len(out)) + " rows")
print("Segment breakdown: " + str(out['segment'].value_counts().to_dict()))

22:43:47  first_purchase_date: 30030 values could not be parsed and became NaT
22:43:47  customer_profiles  segment: 15885 rows have Wholesale? value, kept as-is, needs business confirmation


  WARNING first_purchase_date: 30030 values could not be parsed and became NaT
  WARNING segment: 15885 rows have segment = 'Wholesale?'
          These are kept as-is. A business person must confirm whether
          these are real wholesale customers or a data entry mistake.


22:43:48  customer_profiles  clean: 80000 rows saved



Saved: /content/drive/MyDrive/AmazinChoices/datasets_clean/customer_profiles.csv
Clean: 80000 rows
Segment breakdown: {'New': 16097, 'Churn Risk': 16038, 'Returning': 16020, 'VIP': 15960, 'Wholesale?': 15885}


## Clean customer_interactions

######   **PROBLEM 1:** customer_email has inconsistent casing
######   - FIX:  normalize_email() converts to lowercase

######   **PROBLEM 2:** interaction_timestamp has mixed date formats
######   - FIX: parse_date() handles all formats

######   **PROBLEM 3:** channel and type have leading/trailing whitespace
######   - FIX: .str.upper()

###### This table is used later for:
######   - Identifying customers who complained (churn risk signal)
######   - Measuring support team workload by channel

In [ ]:
df_raw = pd.read_csv(os.path.join(RAW_DIR, 'customer_interactions.csv'))
log.info("customer_interactions  raw: " + str(len(df_raw)) + " rows")
print("Raw: " + str(len(df_raw)) + " rows")

22:43:50  customer_interactions  raw: 120000 rows


Raw: 120000 rows


In [ ]:
out = df_raw.copy()

out['customer_email']        = normalize_email(df_raw['customer_email'])
out['interaction_timestamp'] = parse_date(df_raw['interaction_timestamp'], 'interaction_timestamp')
out['channel']               = df_raw['channel'].str.strip()
out['type']                  = df_raw['type'].str.strip()

out.to_csv(os.path.join(CLEAN_DIR, 'customer_interactions.csv'), index=False)
customer_interactions_clean = out

log.info("customer_interactions  clean: " + str(len(out)) + " rows saved")
print("")
print("Saved: " + CLEAN_DIR + "/customer_interactions.csv")
print("Clean: " + str(len(out)) + " rows")
print("Channels: " + str(out['channel'].value_counts().to_dict()))

22:43:52  interaction_timestamp: 60032 values could not be parsed and became NaT


  WARNING interaction_timestamp: 60032 values could not be parsed and became NaT


22:43:54  customer_interactions  clean: 120000 rows saved



Saved: /content/drive/MyDrive/AmazinChoices/datasets_clean/customer_interactions.csv
Clean: 120000 rows
Channels: {'Email': 24265, 'Social': 24046, 'Phone': 23934, 'Chat': 23929, 'SMS': 23826}


##  Clean refunds

######   **PROBLEM 1:** refund_timestamp has mixed date formats
######   - FIX:  parse_date()

######   **PROBLEM 2:** refund_amount_raw has currency symbols
######   - FIX: clean_currency(), then .clip(lower=0)
######   - WHY:       a refund amount cannot be negative

######   **PROBLEM 3:** reason column has empty strings and inconsistent casing
######   - FIX: .str.strip().str.title() and replace '' with NaN
######   - WHY:       without Title Case, 'damaged' and 'Damaged' and 'DAMAGED' are three separate groups in GROUP BY, splitting what should be one category into three

###### **Business finding documented here** (not a data quality problem):
######  + The refund rate is much higher than the 5-15% industry average.
######  + This is a business problem, not a data problem. We document it.

In [ ]:
df_raw = pd.read_csv(os.path.join(RAW_DIR, 'refunds.csv'))
log.info("refunds  raw: " + str(len(df_raw)) + " rows")
print("Raw: " + str(len(df_raw)) + " rows")

22:43:56  refunds  raw: 80000 rows


Raw: 80000 rows


In [ ]:
out = df_raw.copy()

out['refund_date']   = parse_date(df_raw['refund_timestamp'], 'refund_timestamp')
out['refund_amount'] = clean_currency(df_raw['refund_amount_raw'], 'refund_amount_raw')
out['refund_amount'] = out['refund_amount'].clip(lower=0)

out['reason'] = (
    df_raw['reason']
    .str.strip()
    .str.title()
    .replace({'': np.nan})
)

zero_refund_count = (out['refund_amount'].fillna(0) == 0).sum()
if zero_refund_count > 0:
    log.warning("refunds  refund_amount: " + str(zero_refund_count) + " rows with $0 refund amount, investigate")
    print("  WARNING refund_amount: " + str(zero_refund_count) + " rows have $0 refund. Could be reversed payments or test data.")

total_refunded = out['refund_amount'].sum()
log.info("refunds  total refunded: $" + str(round(total_refunded)))
print("  OK      total refunded: $" + str(round(total_refunded)))
print("  OK      reason values: " + str(out['reason'].value_counts().to_dict()))

out.drop(columns=['refund_timestamp', 'refund_amount_raw'], inplace=True)

out.to_csv(os.path.join(CLEAN_DIR, 'refunds.csv'), index=False)
refunds_clean = out

log.info("refunds  clean: " + str(len(out)) + " rows saved")
print("")
print("Saved: " + CLEAN_DIR + "/refunds.csv")
print("Clean: " + str(len(out)) + " rows")

22:43:57  refund_timestamp: 50178 values could not be parsed and became NaT


  WARNING refund_timestamp: 50178 values could not be parsed and became NaT


22:43:57  refunds  refund_amount: 8039 rows with $0 refund amount, investigate
22:43:57  refunds  total refunded: $1693717


  WARNING refund_amount: 8039 rows have $0 refund. Could be reversed payments or test data.
  OK      total refunded: $1693717
  OK      reason values: {'Other': 12731, 'Customer Changed Mind': 12648, 'Missing Items': 12545, 'Damaged': 12494, 'Late Delivery': 12461, 'Wrong Item': 12375}


22:43:59  refunds  clean: 80000 rows saved



Saved: /content/drive/MyDrive/AmazinChoices/datasets_clean/refunds.csv
Clean: 80000 rows


##  Clean cost_history

######   **PROBLEM 1:** effective_date_raw has mixed date formats
######   - FIX:  parse_date()

######   **PROBLEM 2:** base_cost_raw and packaging_cost_raw have currency symbols
######   - FIX: clean_currency()

######   **PROBLEM 3:** some base_cost and packaging_cost values are negative
######   - FIX: .abs() to take the absolute value
######   - WHY:  unit cost cannot be negative. These are data entry errors where someone typed a minus sign by mistake.
######  **Note**: we use .abs() here (not .clip()) because we want to recover the intended value, not replace it with zero.

######   **PROBLEM 4** sku_raw has inconsistent casing
######   - FIX: clean_currency()

######   **PROBLEM 5** null rate is very high (around 19%)
######   --> DECISION:  keep all rows, do not drop nulls
######   - FIX: dropping rows would remove cost history for valid SKUs.
######    --> We document the null rate instead.

###### New column:
###### **total_cost = base_cost + packaging_cost**
###### fillna(0) means a NULL **base_cost OR packaging_cost** is treated as zero, so total_cost never becomes NULL (which would break gross margin calculations)

In [ ]:
df_raw = pd.read_csv(os.path.join(RAW_DIR, 'cost_history.csv'))
log.info("cost_history  raw: " + str(len(df_raw)) + " rows")
print("Raw: " + str(len(df_raw)) + " rows")

22:44:00  cost_history  raw: 100000 rows


Raw: 100000 rows


In [ ]:
print("Null percentages in raw data:")
null_pct_raw = df_raw.isnull().mean().mul(100).round(1)
for col, pct in null_pct_raw[null_pct_raw > 0].items():
    print("  " + col + ": " + str(pct) + "%")

Null percentages in raw data:
  sku_raw: 2.9%
  base_cost_raw: 9.8%
  packaging_cost_raw: 11.9%
  finance_note: 73.3%


In [ ]:
out = df_raw.copy()

out['sku'] = df_raw['sku_raw'].str.strip().str.upper()
out['effective_date'] = parse_date(df_raw['effective_date_raw'], 'effective_date_raw')
out['base_cost'] = clean_currency(df_raw['base_cost_raw'], 'base_cost_raw')
out['packaging_cost'] = clean_currency(df_raw['packaging_cost_raw'], 'packaging_cost_raw')

#Handle negative cost
for cost_col in ['base_cost', 'packaging_cost']:
    neg_count = (out[cost_col] < 0).sum()
    if neg_count > 0:
        log.warning("cost_history " + cost_col + ": " + str(neg_count) + " negative values converted to absolute value")
        print("WARNING " + cost_col + ": " + str(neg_count) + " negative values converted to absolute value using .abs()")
        out[cost_col] = out[cost_col].abs()
#Calculate total_cost
out['total_cost'] = out['base_cost'].fillna(0) + out['packaging_cost'].fillna(0)

#Drop raw columns
out.drop(
    columns=['sku_raw', 'effective_date_raw', 'base_cost_raw', 'packaging_cost_raw'],
    inplace=True
)

out = out[
    [
        'sku',
        'effective_date',
        'base_cost',
        'packaging_cost',
        'finance_note',
        'total_cost'
    ]
]

#Save
out.to_csv(os.path.join(CLEAN_DIR, 'cost_history.csv'), index=False)
cost_history_clean = out

log.info("cost_history clean: " + str(len(out)) + " rows saved, avg total_cost=" + str(round(out['total_cost'].mean(), 2)))

print("")
print("Saved: " + CLEAN_DIR + "/cost_history.csv")
print("Clean: " + str(len(out)) + " rows")
print("Unique SKUs: " + str(out['sku'].nunique()))
print("Average total_cost: $" + str(round(out['total_cost'].mean(), 2)))

  OK      effective_date_raw: all dates parsed successfully


22:44:01  cost_history base_cost: 1522 negative values converted to absolute value
22:44:01  cost_history packaging_cost: 1492 negative values converted to absolute value


WARNING base_cost: 1522 negative values converted to absolute value using .abs()
WARNING packaging_cost: 1492 negative values converted to absolute value using .abs()


22:44:03  cost_history clean: 100000 rows saved, avg total_cost=4.94



Saved: /content/drive/MyDrive/AmazinChoices/datasets_clean/cost_history.csv
Clean: 100000 rows
Unique SKUs: 56140
Average total_cost: $4.94


##  Clean product_master

######   **PROBLEM 1:** sku has inconsistent casing
######   - FIX:  parse_date()

######   **PROBLEM 2:** product_name and category have inconsistent casing
######   - FIX: .str.title() for consistent display in dashboards

######   **PROBLEM 3:** launch_date has mixed formats, 62% are unparseable
######   - FIX: parse_date() handles what it can, the rest become NaT
######  --> **DECISION:**  do not drop rows where launch_date is NaT
######   - WHY:  the SKU itself is valid even if we do not know the launch date.
######   Dropping these rows would remove valid products from the dimension.
######   The 62% null rate in launch_date was confirmed in the audit as a raw data limitation, not a cleaning error.

######   **PROBLEM 4** ome SKUs appear more than once
######   - FIX: drop_duplicates(subset='sku', keep='first')
######   - WHY: product_master is a dimension table. One row per SKU is required
######   **Duplicate SKUs cause incorrect row counts in JOIN results**

In [ ]:
df_raw = pd.read_csv(os.path.join(RAW_DIR, 'product_master.csv'))
log.info("product_master  raw: " + str(len(df_raw)) + " rows")
print("Raw: " + str(len(df_raw)) + " rows")

22:44:04  product_master  raw: 80000 rows


Raw: 80000 rows


In [ ]:
out = df_raw.copy()

out['sku'] = df_raw['sku'].str.strip().str.upper()
out['product_name'] = df_raw['product_name'].str.strip().str.title()
out['category'] = df_raw['category'].str.strip().str.title()
out['launch_date'] = parse_date(df_raw['launch_date'], 'launch_date')
out['notes'] = df_raw['notes'].astype(str).str.strip()

#Remove duplicate SKUs (SKU is business key)
rows_before_dedup = len(out)
out.drop_duplicates(subset='sku', keep='first', inplace=True)
rows_removed = rows_before_dedup - len(out)

if rows_removed > 0:
    log.warning("product_master sku: " + str(rows_removed) + " duplicate SKU rows removed")
    print("WARNING sku: " + str(rows_removed) + " duplicate rows removed (kept first occurrence)")
#Drop raw columns
out = out[
    [
        'sku',
        'product_name',
        'category',
        'launch_date',
        'notes'
    ]
]

out.to_csv(os.path.join(CLEAN_DIR, 'product_master.csv'), index=False)
product_master_clean = out

#Logging summary
null_launch_pct = round(out['launch_date'].isna().mean() * 100, 1)

log.info(
    "product_master clean: "
    + str(len(out))
    + " rows, "
    + str(null_launch_pct)
    + "% launch_date null"
)

print("")
print("Saved: " + CLEAN_DIR + "/product_master.csv")
print("Clean: " + str(len(out)) + " rows")
print("Unique SKUs: " + str(out['sku'].nunique()))
print("Categories: " + str(out['category'].value_counts().to_dict()))
print("launch_date null: " + str(null_launch_pct) + "% (expected based on raw audit)")

22:44:06  launch_date: 49865 values could not be parsed and became NaT


  WARNING launch_date: 49865 values could not be parsed and became NaT


22:44:08  product_master clean: 80000 rows, 62.3% launch_date null



Saved: /content/drive/MyDrive/AmazinChoices/datasets_clean/product_master.csv
Clean: 80000 rows
Unique SKUs: 80000
Categories: {'Drink Mix': 16098, 'Coffee': 16091, 'Bundle': 15978, 'Snack': 15946, 'Gift Set': 15887}
launch_date null: 62.3% (expected based on raw audit)


##  Clean promotions_log

######   **PROBLEM 1:** start_date_raw and end_date_raw have mixed date formats
######   - FIX:  parse_date()

######   **PROBLEM 2:** promo_code_raw has leading/trailing whitespace
######   - FIX: .str.strip().str.upper() and replace '' with NaN
######   - WHY: this is the JOIN key to orders.promo_code, must match exactly

######   **PROBLEM 3:** discount_pct_raw has currency-style formatting (e.g. '15%')
######   - FIX: clean_currency() strips the percent sign and converts to float

###### **Sanity check performed (not a fix, just a flag):**
###### - Some rows have end_date earlier than start_date.
###### - A promotion cannot end before it starts. This is a raw data problem that needs business clarification. We log it but do not drop the rows.



In [ ]:
df_raw = pd.read_csv(os.path.join(RAW_DIR, 'promotions_log.csv'))
log.info("promotions_log  raw: " + str(len(df_raw)) + " rows")
print("Raw: " + str(len(df_raw)) + " rows")

00:59:12  promotions_log  raw: 85000 rows


Raw: 85000 rows


In [ ]:
out = df_raw.copy()

out['promo_code']   = df_raw['promo_code_raw'].str.strip().str.upper().replace({'': np.nan})
out['start_date']   = parse_date(df_raw['start_date_raw'], 'start_date_raw')
out['end_date']     = parse_date(df_raw['end_date_raw'],   'end_date_raw')
out['discount_pct'] = clean_currency(df_raw['discount_pct_raw'], 'discount_pct_raw')

both_valid     = out['start_date'].notna() & out['end_date'].notna()
backwards_mask = both_valid & (out['end_date'] < out['start_date'])
swap_count     = backwards_mask.sum()

if swap_count > 0:
    temp = out.loc[backwards_mask, 'start_date'].copy()
    out.loc[backwards_mask, 'start_date'] = out.loc[backwards_mask, 'end_date']
    out.loc[backwards_mask, 'end_date']   = temp
    log.info("promotions_log  " + str(swap_count) + " rows: start_date and end_date swapped (export column swap confirmed by diagnostic)")
    print("  FIX   " + str(swap_count) + " rows: start_date and end_date swapped (export column swap)")

still_backwards = (out.loc[both_valid, 'end_date'] < out.loc[both_valid, 'start_date']).sum()
if still_backwards == 0:
    print("  OK    all date ranges are now valid (end_date >= start_date)")
else:
    print("  WARN  " + str(still_backwards) + " rows still have end_date < start_date after fix")

# Drop raw columns
out.drop(columns=['start_date_raw', 'end_date_raw', 'promo_code_raw', 'discount_pct_raw'], inplace=True)

# Reorder columns
out = out[['promo_id', 'promo_code', 'start_date', 'end_date', 'discount_pct', 'notes']]

# Save
out.to_csv(os.path.join(CLEAN_DIR, 'promotions_log.csv'), index=False)
promotions_log_clean = out

log.info("promotions_log  clean: " + str(len(out)) + " rows saved")
print("")
print("Saved: " + CLEAN_DIR + "/promotions_log.csv")
print("Clean: " + str(len(out)) + " rows")
print("Unique promo codes: " + str(out['promo_code'].nunique()))
print("Average discount: " + str(round(out['discount_pct'].mean(), 1)) + "%")

  OK      start_date_raw: all dates parsed successfully


00:59:17  end_date_raw: 32028 values could not be parsed and became NaT
00:59:17  promotions_log  26566 rows: start_date and end_date swapped (export column swap confirmed by diagnostic)


  WARNING end_date_raw: 32028 values could not be parsed and became NaT
  FIX   26566 rows: start_date and end_date swapped (export column swap)
  OK    all date ranges are now valid (end_date >= start_date)


00:59:18  promotions_log  clean: 85000 rows saved



Saved: /content/drive/MyDrive/AmazinChoices/datasets_clean/promotions_log.csv
Clean: 85000 rows
Unique promo codes: 5
Average discount: 15.0%


##  Clean purchase_orders

######   **PROBLEM 1:** created_date has mixed date formats
######   - FIX:  parse_date()

######   **PROBLEM 2:** sku_raw has inconsistent casing
######   - FIX: .str.upper()
######   - WHY: this is the JOIN key to orders.promo_code, must match exactly

######   **PROBLEM 3:** qty_ordered_raw is stored as a string
######   - FIX: pd.to_numeric() then convert to integer

######   **PROBLEM 4:** unit_cost_raw has currency symbols and negative values
######   - FIX: clean_currency() then .abs() on negative values
######   - WHY: we use .abs() (not .clip()) because the intended value was probably positive and the minus sign was typed by mistake.
###### --> A unit_cost of -5.00 probably means 5.00.

######   **PROBLEM 5:** status and supplier_name have inconsistent casing
######   - FIX: .str.strip().str.title()

######  New column:
######    --> **total_po_cost = qty_ordered * unit_cost**
######    This is the total cost of each purchase order line


In [ ]:
df_raw = pd.read_csv(os.path.join(RAW_DIR, 'purchase_orders.csv'))
log.info("purchase_orders  raw: " + str(len(df_raw)) + " rows")
print("Raw: " + str(len(df_raw)) + " rows")

22:44:17  purchase_orders  raw: 90000 rows


Raw: 90000 rows


In [ ]:
out = df_raw.copy()

out['po_id'] = df_raw['po_id']
out['created_date'] = parse_date(df_raw['created_date'], 'created_date')
out['sku']          = df_raw['sku_raw'].astype(str).str.strip().str.upper()

out['supplier_name'] = df_raw['supplier_name'].str.strip().str.title()
out['qty_ordered']  = pd.to_numeric(df_raw['qty_ordered_raw'], errors='coerce').astype('Int64')
out['unit_cost']    = clean_currency(df_raw['unit_cost_raw'], 'unit_cost_raw')
out['status']        = df_raw['status'].str.strip().str.title()
out['warehouse_note'] = df_raw['warehouse_note'].astype(str).str.strip()

#Handle negative unit_cost
negative_cost_mask  = out['unit_cost'] < 0
negative_cost_count = negative_cost_mask.sum()

if negative_cost_count > 0:
    log.warning("purchase_orders  unit_cost: " + str(negative_cost_count) + " negative values converted to absolute value")
    print("  WARNING unit_cost: " + str(negative_cost_count) + " negative values converted to absolute value using .abs()")
    print("          Sample PO IDs: " + str(out.loc[negative_cost_mask, 'po_id'].head(3).tolist()))
    out.loc[negative_cost_mask, 'unit_cost'] = out.loc[negative_cost_mask, 'unit_cost'].abs()

#Calculate total_po_cost safely
out['total_po_cost'] = out['qty_ordered'] * out['unit_cost']

#Drop raw columns
out.drop(columns=['sku_raw', 'qty_ordered_raw', 'unit_cost_raw'], inplace=True)

#Reorder columns
out = out[
    [
        'po_id',
        'created_date',
        'sku',
        'supplier_name',
        'qty_ordered',
        'unit_cost',
        'total_po_cost',
        'status',
        'warehouse_note'
    ]
]

out.to_csv(os.path.join(CLEAN_DIR, 'purchase_orders.csv'), index=False)
purchase_orders_clean = out

log.info("purchase_orders  clean: " + str(len(out)) + " rows, total_po_cost=$" + str(round(out['total_po_cost'].sum())))
print("")
print("Saved: " + CLEAN_DIR + "/purchase_orders.csv")
print("Clean: " + str(len(out)) + " rows")
print("Status breakdown: " + str(out['status'].value_counts().to_dict()))


  OK      created_date: all dates parsed successfully


22:44:21  purchase_orders  unit_cost: 1384 negative values converted to absolute value


  WARNING unit_cost: 1384 negative values converted to absolute value using .abs()
          Sample PO IDs: ['PO-2026-000010', 'PO-2025-000177', 'PO-2025-000184']


22:44:23  purchase_orders  clean: 90000 rows, total_po_cost=$223603132



Saved: /content/drive/MyDrive/AmazinChoices/datasets_clean/purchase_orders.csv
Clean: 90000 rows
Status breakdown: {'Received': 22601, 'Cancelled': 22522, 'Open': 22459, 'Partially Received': 22418}


##  Clean marketing_spend

######   **PROBLEM 1:** log_timestamp has mixed date formats
######   - FIX:  parse_date()

######   **PROBLEM 2:** spend_raw has currency symbols and negative values
######   - FIX: clean_currency() then .clip(lower=0)
######   - WHY: spending cannot be negative

######   **PROBLEM 3:** clicks_raw has US thousands commas (e.g. '1,347')
######   - FIX: remove commas then convert to integer

###### **IMPORTANT ANOMALY** - do not use this table for ROAS analysis:
######   The **total spend** in this table is **much larger than total revenue.**
######   Possible causes: spend recorded in a different unit (impressions?),  cumulative totals mixed with daily rows, wrong currency, or test data.
######   We document this anomaly in the log and exclude this table from any return-on-ad-spend calculations until Finance clarifies the unit.
######   **CPC (cost per click)** comparisons between platforms are still valid.

In [ ]:
df_raw = pd.read_csv(os.path.join(RAW_DIR, 'marketing_spend.csv'))
log.info("marketing_spend  raw: " + str(len(df_raw)) + " rows")
print("Raw: " + str(len(df_raw)) + " rows")

00:57:35  marketing_spend  raw: 90000 rows


Raw: 90000 rows


In [ ]:
out = df_raw.copy()

out['log_timestamp'] = parse_date(df_raw['log_timestamp'], 'log_timestamp')

out['ad_platform'] = (df_raw['ad_platform'].astype(str).str.strip().str.title())

out['campaign_name'] = (df_raw['campaign_name_raw'].astype(str).str.strip())

out['target_platform'] = (df_raw['target_platform_raw'].astype(str).str.strip().replace({'': np.nan}))

out['spend'] = clean_currency(df_raw['spend_raw'], 'spend_raw')

out['clicks'] = pd.to_numeric(
    df_raw['clicks_raw']
        .astype(str)
        .str.replace(',', '', regex=False)
        .str.strip(),
    errors='coerce'
).astype('Int64')

out['notes'] = df_raw['notes'].astype(str).str.strip()

#Handle negative spend
negative_spend_count = (out['spend'] < 0).sum()

if negative_spend_count > 0:
    log.warning(
        "marketing_spend spend: "
        + str(negative_spend_count)
        + " negative values clipped to 0"
    )
    print("WARNING spend:", negative_spend_count,
          "negative values clipped to 0")

out['spend'] = out['spend'].clip(lower=0)

#Optional: negative clicks protection
neg_clicks = (out['clicks'] < 0).sum()
if neg_clicks > 0:
    log.warning("marketing_spend clicks: negative values set to NULL")
    out.loc[out['clicks'] < 0, 'clicks'] = pd.NA

#Anomaly check only if ratio unreasonable
total_spend = out['spend'].sum()
estimated_revenue = 2_100_000

if total_spend > estimated_revenue * 2:
    print("")
    print("ANOMALY WARNING")
    print("Total spend: $", round(total_spend))
    print("Revenue reference: ~$2,100,000")
    print("Spend exceeds reasonable ratio.")
    print("Exclude from ROAS until unit confirmed.")

    log.warning(
        "marketing_spend anomaly: spend $"
        + str(round(total_spend))
        + " >> revenue reference"
    )

#Drop raw columns
out.drop(
    columns=[
        'campaign_name_raw',
        'target_platform_raw',
        'spend_raw',
        'clicks_raw'
    ],
    inplace=True
)

#Reorder columns
out = out[
    [
        'log_timestamp',
        'ad_platform',
        'campaign_name',
        'target_platform',
        'spend',
        'clicks',
        'notes'
    ]
]

out.to_csv(os.path.join(CLEAN_DIR, 'marketing_spend.csv'), index=False)
marketing_spend_clean = out

log.info("marketing_spend clean: " + str(len(out)) + " rows saved")

print("")
print("Saved:", CLEAN_DIR + "/marketing_spend.csv")
print("Clean:", len(out), "rows")
print("Platform breakdown:", out['ad_platform'].value_counts().to_dict())
print("Total Spend: $", round(total_spend, 2))



  OK      log_timestamp: all dates parsed successfully


00:58:15  marketing_spend spend: 1305 negative values clipped to 0
00:58:15  marketing_spend anomaly: spend $305185119 >> revenue reference


WARNING spend: 1305 negative values clipped to 0

ANOMALY WARNING
Total spend: $ 305185119
Revenue reference: ~$2,100,000
Spend exceeds reasonable ratio.
Exclude from ROAS until unit confirmed.


00:58:17  marketing_spend clean: 90000 rows saved



Saved: /content/drive/MyDrive/AmazinChoices/datasets_clean/marketing_spend.csv
Clean: 90000 rows
Platform breakdown: {'Google Ads': 22869, 'Instagram Ads': 22413, 'Tiktok Ads': 22402, 'Facebook Ads': 22316}
Total Spend: $ 305185119.38


##  Clean customer_interactions

######   **PROBLEM 1:** effective_date_raw has mixed date formats
######   - FIX:  parse_date()

######   **PROBLEM 2:** min_weight_lbs_raw and max_weight_lbs_raw have mixed formats
######   - FIX: clean_currency() removes symbols, converts to float

######   **PROBLEM 3:** rate_raw has currency symbols and potentially negative rates
######   - FIX: clean_currency() then .clip(lower=0)

######   **PROBLEM 4:** carrier has inconsistent casing
######   - FIX: .str.title()

In [ ]:
df_raw = pd.read_csv(os.path.join(RAW_DIR, 'shipping_rates.csv'))
log.info("shipping_rates  raw: " + str(len(df_raw)) + " rows")
print("Raw: " + str(len(df_raw)) + " rows")

22:44:34  shipping_rates  raw: 80000 rows


Raw: 80000 rows


In [ ]:
out = df_raw.copy()

out['effective_date'] = parse_date(df_raw['effective_date_raw'], 'effective_date_raw')
out['min_weight_lbs'] = clean_currency(df_raw['min_weight_lbs_raw'], 'min_weight_lbs_raw')
out['max_weight_lbs'] = clean_currency(df_raw['max_weight_lbs_raw'], 'max_weight_lbs_raw')
out['rate']           = clean_currency(df_raw['rate_raw'], 'rate_raw').clip(lower=0)
out['zone']           = df_raw['zone'].str.strip()

out.drop(columns=['effective_date_raw', 'min_weight_lbs_raw', 'max_weight_lbs_raw', 'rate_raw'], inplace=True)

out = out[
    [
        'effective_date',
        'carrier',
        'zone',
        'min_weight_lbs',
        'max_weight_lbs',
        'rate',
        'notes'
    ]
]
out.to_csv(os.path.join(CLEAN_DIR, 'shipping_rates.csv'), index=False)
shipping_rates_clean = out

log.info("shipping_rates  clean: " + str(len(out)) + " rows saved")
print("")
print("Saved: " + CLEAN_DIR + "/shipping_rates.csv")
print("Clean: " + str(len(out)) + " rows")
print("Carriers: " + str(out['carrier'].value_counts().to_dict()))

22:44:35  effective_date_raw: 50175 values could not be parsed and became NaT


  WARNING effective_date_raw: 50175 values could not be parsed and became NaT


22:44:37  shipping_rates  clean: 80000 rows saved



Saved: /content/drive/MyDrive/AmazinChoices/datasets_clean/shipping_rates.csv
Clean: 80000 rows
Carriers: {'FedEx': 20202, 'USPS': 20049, 'UPS': 19928, 'DHL': 19821}


## Print cleaning summary and save to file
###### This final cell shows a summary table of all 11 clean files.
###### It also saves a JSON file that Notebook 03 uses to check expected values.

In [ ]:
ALL_TABLE_NAMES = [
    'orders', 'order_items', 'customer_profiles', 'customer_interactions',
    'marketing_spend', 'refunds', 'cost_history', 'product_master',
    'promotions_log', 'purchase_orders', 'shipping_rates'
]

summary_data = {}
total_clean_rows = 0

print("")
print("Table".ljust(30) + "Rows".rjust(9) + "Cols".rjust(6) + "Null%".rjust(7) + "MB".rjust(6))
print("-" * 60)

for table_name in ALL_TABLE_NAMES:
    file_path = os.path.join(CLEAN_DIR, table_name + '.csv')
    if os.path.exists(file_path):
        df       = pd.read_csv(file_path)
        rows     = len(df)
        cols     = df.shape[1]
        null_pct = round(df.isnull().sum().sum() / df.size * 100, 1)
        mb       = round(os.path.getsize(file_path) / 1024 / 1024, 1)
        total_clean_rows += rows
        print(table_name.ljust(30) + str(rows).rjust(9) + str(cols).rjust(6) + (str(null_pct) + "%").rjust(7) + (str(mb) + "M").rjust(6))
        summary_data[table_name] = {'rows': rows, 'cols': cols, 'null_pct': null_pct, 'mb': mb}
    else:
        print(table_name.ljust(30) + "  MISSING")

print("-" * 60)
print("TOTAL".ljust(30) + str(total_clean_rows).rjust(9))

summary_output = {
    'cleaned_at':      str(datetime.now()),
    'clean_dir':       CLEAN_DIR,
    'total_rows':      total_clean_rows,
    'tables':          summary_data,
}

summary_path = os.path.join(AUDIT_DIR, '02_cleaning_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary_output, f, indent=2)

print("")
print("Files saved:")
print("  " + summary_path)
print("  " + LOG_PATH)


Table                              Rows  Cols  Null%    MB
------------------------------------------------------------
orders                           110000    12  10.6% 15.0M
order_items                      130000     9  12.9% 13.5M
customer_profiles                 80000     6  16.6%  5.5M
customer_interactions            120000     6  18.0% 10.7M
marketing_spend                   90000     7  15.5%  6.4M
refunds                           80000     6  23.6%  7.1M
cost_history                     100000     6  16.3%  5.9M
product_master                    80000     5  19.6%  4.7M
promotions_log                    85000     6  21.9%  6.6M
purchase_orders                   90000     9  11.1%  9.4M
shipping_rates                    80000     7  22.7%  3.0M
------------------------------------------------------------
TOTAL                           1045000

Files saved:
  /content/drive/MyDrive/AmazinChoices/audit/02_cleaning_summary.json
  /content/drive/MyDrive/AmazinChoices/audit/